In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, freqz
from ipywidgets import SelectionSlider, HTML, HBox, Layout
from IPython.display import display

# ============================================================
# IIR OPTIMIZATION — LP OBJECTIVE FUNCTIONS
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.op-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.op-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.op-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14.5px;
    line-height:1.45;
    margin-bottom:7px;
}

.op-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:7px;
    font-size:14px;
    line-height:1.45;
}

.op-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:14.5px;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="op-root">

<div class="op-header">
IIR Optimization — Objective Functions and Lp Error Measures
</div>

<div class="op-doc">

Optimization-based IIR design determines the filter parameter vector <b>ξ</b>
by minimizing an objective function constructed from the frequency-domain
approximation error

<div style="text-align:center;font-size:15px;margin:5px 0;">
<b>
e(ξ,ω) = H<sub>r</sub>(ξ,ω) - H<sub>dr</sub>(ω).
</b>
</div>

After sampling this error at K frequencies, different norms lead to different
optimization criteria:

<div style="text-align:center;font-size:15px;margin:5px 0;">
<b>
L₁ = Σ|eᵢ|,
&nbsp;&nbsp;
L₂ = √Σ|eᵢ|²,
&nbsp;&nbsp;
L∞ = max|eᵢ|.
</b>
</div>

The left plot shows one fixed filter approximation. Therefore, it does not
change when p changes.

The right plot shows the <b>relative contribution of each error sample to the
Lp criterion</b>. As p increases, small errors become progressively less
important and the largest errors dominate. In the limit
<b>p → ∞</b>, only the maximum error determines the criterion.

</div>

</div>
"""))

# ============================================================
# EXAMPLE RESPONSE
# ============================================================

b,a = butter(4,0.45)

omega,H = freqz(b,a,worN=2048)

wn = omega/np.pi

actual = np.abs(H)

desired = np.where(wn <= 0.45,1.0,np.where(wn >= 0.55,0.0,np.nan))

mask = (wn <= 0.45) | (wn >= 0.55)

error = actual[mask]-desired[mask]

error_abs = np.abs(error)

error_max = np.max(error_abs)

wn_error = wn[mask]

# ============================================================
# CONTROL
# ============================================================

p_slider = SelectionSlider(options=[1,2,4,8,16,32,'∞'],value=2,description='p:',continuous_update=True,style={'description_width':'20px'},layout=Layout(width='430px'))

info = HTML(layout=Layout(width='450px'))

controls = HBox([p_slider,info],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='7px 10px',margin='0 0 7px 0',align_items='center'))

# ============================================================
# FIGURE — CREATED ONCE
# ============================================================

fig,(ax1,ax2) = plt.subplots(1,2,figsize=(9.0,4.0))

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# LEFT — FIXED DESIRED AND ACTUAL RESPONSES
# ============================================================

ax1.plot(wn,actual,color='red',linewidth=1.4,label='Actual response')

ax1.plot([0,0.45],[1,1],color='black',linewidth=1.2,label='Desired response')

ax1.plot([0.55,1],[0,0],color='black',linewidth=1.2)

ax1.axvspan(0.45,0.55,alpha=0.08)

ax1.set_xlim(0,1)

ax1.set_ylim(0,1.15)

ax1.set_title('Desired and Actual Magnitude Responses')

ax1.set_xlabel(r'Normalized frequency $\omega/\pi$')

ax1.set_ylabel('Magnitude')

ax1.grid(True,linestyle=':',alpha=0.25)

ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# RIGHT — DYNAMIC LP ERROR CONTRIBUTION
# ============================================================

contribution_line, = ax2.plot(wn_error,np.zeros_like(wn_error),color='red',linewidth=1.4)

max_error_marker, = ax2.plot([],[],'o',markersize=5,label='Maximum error')

ax2.set_xlim(0,1)

ax2.set_ylim(0,1.05)

ax2.set_title(r'Relative Contribution to the $L_p$ Criterion')

ax2.set_xlabel(r'Normalized frequency $\omega/\pi$')

ax2.set_ylabel('Relative contribution')

ax2.grid(True,linestyle=':',alpha=0.25)

ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),frameon=False)

plt.subplots_adjust(left=0.08,right=0.98,top=0.91,bottom=0.20,wspace=0.28)

# ============================================================
# UPDATE
# ============================================================

def update(change=None):

    p = p_slider.value

    if p == '∞':

        objective = error_max

        contribution = np.zeros_like(error_abs)

        max_index = np.argmax(error_abs)

        contribution[max_index] = 1.0

        label = 'L∞ = max |eᵢ|'

    else:

        objective = np.sum(error_abs**p)**(1.0/p)

        contribution = (error_abs/error_max)**p

        max_index = np.argmax(error_abs)

        label = f'L{p}'

    contribution_line.set_data(wn_error,contribution)

    max_error_marker.set_data([wn_error[max_index]],[1.0])

    info.value = f"""
    <div style="font-size:14px;">
    <b>{label}</b>
    &nbsp;&nbsp;
    Objective = <b>{objective:.6f}</b>
    </div>
    """

    fig.canvas.draw_idle()

# ============================================================
# EVENTS
# ============================================================

p_slider.observe(update,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(controls)

display(fig.canvas)

# ============================================================
# INITIAL UPDATE
# ============================================================

update()